In [ ]:
from google.colab import files
uploaded=files.upload()
print(uploaded)

Saving deaths.xlsx to deaths (2).xlsx
Saving pumps.xlsx to pumps (1).xlsx
{'deaths (2).xlsx': b'PK\x03\x04\x14\x00\x06\x00\x08\x00\x00\x00!\x00b\xee\x9dh^\x01\x00\x00\x90\x04\x00\x00\x13\x00\x08\x02[Content_Types].xml \xa2\x04\x02(\xa0\x00\x02\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\

In [ ]:
import pandas as pd
deaths = pd.read_excel('deaths (1).xlsx')
pumps = pd.read_excel('pumps.xlsx')
display(deaths)

,death_count,x_latitude,y_longitude
0,1,51.513418,-0.137930
1,1,51.513418,-0.137930
2,1,51.513418,-0.137930
3,1,51.513361,-0.137883
4,1,51.513361,-0.137883
...,...,...,...
484,1,51.514706,-0.137065
485,1,51.514706,-0.137065
486,1,51.512311,-0.138474
487,1,51.511998,-0.138123


In [ ]:
# Select only coordinate columns
locations_deaths = deaths[['x_latitude', 'y_longitude']]

# Split the single combined column in 'pumps' into separate columns
if 'pump_name,x_coordinate,y_coordinate' in pumps.columns:
    split_data = pumps['pump_name,x_coordinate,y_coordinate'].str.split(',', expand=True)
    split_data.columns = ['pump_name', 'x_coordinate', 'y_coordinate']
    pumps = pd.concat([pumps.drop(columns=['pump_name,x_coordinate,y_coordinate']), split_data], axis=1)

    # Convert coordinate columns to numeric type
    pumps['x_coordinate'] = pd.to_numeric(pumps['x_coordinate'])
    pumps['y_coordinate'] = pd.to_numeric(pumps['y_coordinate'])

locations_pumps = pumps[['x_coordinate', 'y_coordinate']]

# Convert to list format for mapping
deaths_list = locations_deaths.values.tolist()
pumps_list = locations_pumps.values.tolist()

# Import folium
import folium

map = folium.Map(location=[51.5132119,-0.13666], tiles='Cartodb Positron', zoom_start=17)

for point in range(0, len(locations_deaths)):
    folium.CircleMarker(
        deaths_list[point],
        radius=8,
        color='red',
        fill=True,
        fill_color='black',
        opacity=0.4
    ).add_to(map)
map

In [ ]:
from folium.plugins import FastMarkerCluster
FastMarkerCluster(data=list(zip(deaths['x_latitude'].values, deaths['y_longitude'].values))).add_to(map)
folium.LayerControl().add_to(map)
map

In [ ]:
for index, row in deaths.iterrows():

    folium.CircleMarker(location=(row["x_latitude"],
                                  row["y_longitude"]),
                        radius= row['death_count']*10,
                        color='#FA8072',
                        fillOpacity=0.9,
                        fill=True).add_to(map)
map

In [ ]:
base_map = folium.Map(location=[51.5132119,-0.13666], tiles='https://watercolormaps.collection.cooperhewitt.org/tile/watercolor/{z}/{x}/{y}.jpg',attr='Map tiles by Stamen Design, under CC BY 3.0. Data by OpenStreetMap, under CC BY SA.' , zoom_start=16)

for i,row in pumps.iterrows():
    folium.Marker([row['x_coordinate'],row['y_coordinate']], popup=row['pump_name']).add_to(base_map)
base_map
heat_data = [[row['x_latitude'],row['y_longitude']] for index, row in locations_deaths.iterrows()]

from folium import plugins
from folium.plugins import HeatMap
HeatMap(heat_data).add_to(base_map)
base_map